# BENZI — LoRA fine-tune on Colab

**Open only this link (GitHub `main`):**  
https://colab.research.google.com/github/sameedsaeed123/final-year-benzi/blob/main/fyp-ml-demos/finetune/BENZI_Colab_Train.ipynb

1. **Runtime → Change runtime type → T4 GPU**
2. **Runtime → Run all** (do not use an old saved copy of this notebook)

| Step | Time |
|------|------|
| Setup + pip | ~3 min |
| Dataset | ~5 min |
| Train Qwen2.5-3B LoRA | ~4 min |
| Merge (CPU save) | **15–40 min — do not press Stop** |
| Download zip | ~2 min |

**Do not disconnect** until the zip downloads.  
Mac setup after download: `benzi-server/docs/COLAB_TRAIN.md`

In [ ]:
# Cell 1 — GPU
!nvidia-smi
import torch
assert torch.cuda.is_available(), "Runtime → Change runtime type → T4 GPU"
print("CUDA OK")

In [ ]:
# Cell 2 — Fresh clone (absolute path; never %cd final-year-benzi/...)
import os
import shutil
import subprocess
import sys
from pathlib import Path

os.chdir("/content")
REPO = Path("/content/final-year-benzi")
ML = REPO / "fyp-ml-demos"
REPO_URL = "https://github.com/sameedsaeed123/final-year-benzi.git"

if REPO.exists():
    shutil.rmtree(REPO)
subprocess.check_call(["git", "clone", "--depth", "1", REPO_URL, str(REPO)])
os.chdir(ML)

subprocess.run([sys.executable, "-m", "pip", "uninstall", "-y", "torchao"], check=False)
subprocess.check_call([
    sys.executable, "-m", "pip", "install", "-q",
    "-r", "requirements.txt",
    "-r", "requirements-finetune-colab.txt",
    "bitsandbytes>=0.43.0", "accelerate", "peft", "datasets",
])

assert (ML / "finetune" / "merge_lora_colab.py").is_file()
print("Working directory:", ML.resolve())
print("Disk free GB:", shutil.disk_usage("/content").free / 1e9)

In [ ]:
# Cell 3 — Dataset (~5 min)
!python finetune/prepare_dataset.py --max-total 1200

In [ ]:
# Cell 4 — Train (~4 min on T4). Wait for: Saved LoRA adapter
!python finetune/train_qlora.py --model Qwen/Qwen2.5-3B-Instruct --max-steps 60 --max-length 384 --batch-size 1 --grad-accum 4

In [ ]:
# Cell 5 — Merge (15–40 min). Uses merge_lora_colab.py — DO NOT STOP
from pathlib import Path
import shutil
import subprocess
import sys

adapter = Path("finetune/adapters/benzi-lora/adapter_config.json")
if not adapter.is_file():
    raise RuntimeError("Training failed — re-run Cell 4 only.")

subprocess.run([sys.executable, "-m", "pip", "uninstall", "-y", "torchao"], check=False)
merged = Path("finetune/merged/benzi-empathetic-hf")
if merged.exists():
    shutil.rmtree(merged)

free_gb = shutil.disk_usage("/content").free / 1e9
print(f"Disk free: {free_gb:.1f} GB (need ~10)")
if free_gb < 10:
    raise RuntimeError("Low disk — Runtime → Disconnect and delete runtime, then Run all from Cell 1.")

!python finetune/merge_lora_colab.py --model Qwen/Qwen2.5-3B-Instruct

In [ ]:
# Cell 6 — Download zip
import shutil
from pathlib import Path
from google.colab import files

merged = Path("finetune/merged/benzi-empathetic-hf")
if not (merged / "config.json").is_file():
    raise RuntimeError("Merge not done — wait for Cell 5 to finish (see Done in ... min).")

shutil.make_archive("/content/benzi-empathetic-trained", "zip", merged)
files.download("/content/benzi-empathetic-trained.zip")
print("Download started: benzi-empathetic-trained.zip")

## On your Mac

```bash
mkdir -p ~/benzi-models && cd ~/benzi-models
unzip ~/Downloads/benzi-empathetic-trained.zip -d benzi-empathetic-hf
cd benzi-empathetic-hf
cat > Modelfile << 'EOF'
FROM .
PARAMETER temperature 0.65
PARAMETER num_ctx 4096
SYSTEM You are BENZI AI — supportive wellness between therapy sessions. Not a therapist. Defer clinical questions to their therapist.
EOF
ollama create benzi-empathetic-trained -f Modelfile
```

In `benzi-server/.env`: `OLLAMA_MODEL=benzi-empathetic-trained` then restart API.

---
**If merge keeps failing:** run only this in a new cell to save the adapter (~100MB), merge on Mac:

```python
import shutil
from google.colab import files
shutil.make_archive("/content/benzi-lora-adapter", "zip", "finetune/adapters/benzi-lora")
files.download("/content/benzi-lora-adapter.zip")
```